In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 Extreme Logistic Regressor with Direct Inverse Class Weighting (`models/lr_feng_esi1_extreme.ipynb`)

This notebook trains a **Binary Logistic Regressor** for **ESI 1 vs Not ESI 1** with **Direct Inverse Class Frequency Weighting**:

### System Architecture & Key Features
1. **Majority Class Undersampling**: Configurable majority class (`"not_1"`) undersampling (e.g. 1:1 ratio or custom percentage).
2. **Feature Set (13 Clinical Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and 10 vital sign anomaly flags.
3. **Direct Inverse Class Frequency Weighting**: Calculates exact inverse class frequency weight $w_{\text{inv}} = N_{\text{total}} / N_{\text{ESI 1}}$ and applies it directly during model training (no multipliers).
4. **Evaluation & CSV Reports**: Reports actual vs. predicted class counts, Precision, Recall, and PR-AUC. Exports `reports/lr_feng_esi1_val_report.csv` and `reports/lr_feng_esi1_test_report.csv`.
5. **Diagnostic Plots**: Saves bar chart (`plots/esi1_metrics_comparison_barchart.png`) and overlaid PR curves (`plots/esi1_pr_curves_overlaid.png`).
6. **Model Artifact Export**: Saved to `deploy/lr_feng_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Compute 13 FE Inputs & Apply Undersampling / Subsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Compute 13 Clinical Feature Engineering flags
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))

initial_rows <- nrow(df_feng)
df_feng <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining: %d)\n", initial_rows - nrow(df_feng), nrow(df_feng)))
# Configurable Class Keep Ratios / Undersampling
keep_ratio_1     <- 1.00   # Keep 100% of ESI 1 rows
keep_ratio_not_1 <- 1.00   # Keep 100% of 'not_1' rows (modify e.g. 0.05 for 5% undersampling)
idx_1     <- which(df_feng$target_layer1 == "1")
idx_not_1 <- which(df_feng$target_layer1 == "not_1")
kept_1     <- sample(idx_1,     size = round(length(idx_1)     * keep_ratio_1))
kept_not_1 <- sample(idx_not_1, size = round(length(idx_not_1) * keep_ratio_not_1))
df_feng <- df_feng[sort(c(kept_1, kept_not_1)), ]
cat(sprintf("Binary ESI 1 Dataset Ready: %d total rows x %d cols\n", nrow(df_feng), ncol(df_feng)))
cat("Binary Target Distribution:\n")
print(table(df_feng$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Continuous Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Calculate Direct Inverse Class Frequency Weight & Train Model (No Multipliers)
# ---------------------------------------------------------
set.seed(config$training$random_state)

n_total <- nrow(train_df)
n_esi1  <- sum(train_df$target_layer1 == "1")
inv_weight_esi1 <- n_total / n_esi1

cat(sprintf("=== Direct Inverse Class Frequency Weight Calculation ===\n"))
cat(sprintf("  - Total Training Rows          : %d\n", n_total))
cat(sprintf("  - ESI 1 Minority Class Count   : %d\n", n_esi1))
cat(sprintf("  - Direct Inverse Class Weight  : %.4f\n\n", inv_weight_esi1))
feat_names <- setdiff(names(train_df), c("target_layer1"))
formula_lr <- as.formula(paste("target_layer1 ~", paste(feat_names, collapse = " + ")))
weights_esi1 <- ifelse(train_df$target_layer1 == "1", inv_weight_esi1, 1.0)
cat("Training Binary ESI 1 Logistic Regressor with Direct Inverse Class Weight...\n")
lr_esi1_model <- multinom(formula_lr, data = train_df, weights = weights_esi1, trace = FALSE, MaxNWts = 5000)
cat("Binary ESI 1 Logistic Regression Training Complete!\n")
print(summary(lr_esi1_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive Evaluation Across Splits & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_and_report_esi1 <- function(model, data, set_name) {
  pred_val <- as.character(predict(model, newdata = data))
  pred_val[is.na(pred_val)] <- "not_1"
  pred_fac <- factor(pred_val, levels = c("1", "not_1"))
  act_fac  <- factor(data$target_layer1, levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  prob_res <- predict(model, newdata = data, type = "probs")
  prob_1   <- if (is.matrix(prob_res)) {
    if ("1" %in% colnames(prob_res)) prob_res[, "1"] else 1 - prob_res[, "not_1"]
  } else {
    1 - prob_res
  }
  pr_auc <- calc_pr_auc(ifelse(act_fac == "1", 1, 0), prob_1)
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = c("1", "not_1"),
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(c(prec, ifelse(is.na(cm$byClass["Neg Pred Value"]), 0, cm$byClass["Neg Pred Value"])), 4),
    Recall       = round(c(rec, ifelse(is.na(cm$byClass["Specificity"]), 0, cm$byClass["Specificity"])), 4),
    PR_AUC       = round(c(pr_auc, NA), 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   BINARY ESI 1 DIRECT CLASS WEIGHTED LR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ESI 1 Precision      : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  ESI 1 Recall (Sens)  : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  ESI 1 F1 Score       : %.4f\n", f1))
  cat(sprintf("  ESI 1 PR-AUC         : %.4f\n", pr_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Class Count Comparison & Performance Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, prec = prec, rec = rec, pr_auc = pr_auc, prob_1 = prob_1, act_fac = act_fac, report_df = report_df))
}
res_train <- evaluate_and_report_esi1(lr_esi1_model, train_df, "Train")
res_val   <- evaluate_and_report_esi1(lr_esi1_model, val_df,   "Validation")
res_test  <- evaluate_and_report_esi1(lr_esi1_model, test_df,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "lr_feng_esi1_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "lr_feng_esi1_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/lr_feng_esi1_val_report.csv\n")
cat("Test CSV Report written to:       reports/lr_feng_esi1_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart & Overlaid PR Curves)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train$acc,  res_val$acc,  res_test$acc),
  Precision = c(res_train$prec, res_val$prec, res_test$prec),
  Recall    = c(res_train$rec,  res_val$rec,  res_test$rec),
  PR_AUC    = c(res_train$pr_auc, res_val$pr_auc, res_test$pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics Comparison (Binary ESI 1 Direct Class Weighted)",
       subtitle = "Comparing Accuracy, Precision, Recall, and PR-AUC across splits",
       y = "Metric Value", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "esi1_metrics_comparison_barchart.png"), plot = p_bar, width = 9, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/esi1_metrics_comparison_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Binary ESI 1 Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "lr_feng_esi1_extreme_model.rds")
saveRDS(list(model = lr_esi1_model, preproc = preproc), file = model_path)
cat("Direct Inverse Class Weighted Binary ESI 1 Logistic Regressor saved to:", model_path, "\n")